# Phase 9: Knowledge Graph

This notebook completes the Phase 9 roadmap tasks:

48. Create a Line → Station → Feature → Failure graph.
49. Build relationships between stations.
50. Calculate centrality metrics.
51. Identify candidate failure propagation routes.
52. Highlight critical nodes.

The graph uses raw-feature metadata plus the production-safe Phase 7 SHAP evidence and Phase 8 process relationships. Candidate propagation routes are observational diagnostic priorities, not causal claims.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
PROJECT_ROOT

## Run The Phase 9 Pipeline

The reusable script loads Phase 2 feature metadata, Phase 7 production-safe SHAP evidence, and Phase 8 station transitions and bottleneck results. It creates graph tables, computes centrality, ranks critical nodes, exports GraphML, and renders a technical HTML report.

In [ ]:
import subprocess

script_path = PROJECT_ROOT / 'src' / 'data' / 'phase9_knowledge_graph.py'
subprocess.run([sys.executable, str(script_path)], cwd=PROJECT_ROOT, check=True)

## Load Graph Outputs

In [ ]:
import pandas as pd
from IPython.display import Image, display

reports = PROJECT_ROOT / 'reports'
figures = reports / 'figures'
nodes = pd.read_csv(reports / 'phase9_knowledge_graph_nodes.csv')
edges = pd.read_csv(reports / 'phase9_knowledge_graph_edges.csv')
centrality = pd.read_csv(reports / 'phase9_station_centrality_metrics.csv')
routes = pd.read_csv(reports / 'phase9_failure_propagation_routes.csv')

print(f'Nodes: {len(nodes):,}')
print(f'Relationships: {len(edges):,}')
display(nodes.groupby('node_type').size().rename('node_count').to_frame())
display(edges.groupby('relationship').size().rename('edge_count').to_frame())

## Line → Station → Feature → Failure Hierarchy

Every raw numeric, categorical, and date feature is connected to its manufacturing station and line. A feature receives a failure edge only when the production-safe model contains non-zero SHAP evidence for that raw feature or its missingness/category derivative.

In [ ]:
failure_edges = edges.query("relationship == 'MODEL_ASSOCIATED_WITH_FAILURE'")
failure_edges.sort_values('failure_evidence', ascending=False).head(20)

## Station Relationships And Centrality

PageRank measures influence through incoming production flow. Betweenness identifies bridge stations along high-volume routes. Closeness measures how readily a station can reach the rest of the observed process network.

In [ ]:
display(centrality.head(20)[['critical_rank','station','critical_node_score','centrality_score','pagerank','betweenness_centrality','bottleneck_score','failure_lift','station_shap_importance']])
display(Image(filename=str(figures / 'phase9_station_relationship_network.png')))

## Critical Nodes

The composite critical-node score combines bottleneck severity (25%), failure lift (20%), station-level SHAP attribution (20%), network centrality (20%), and product volume (15%). This is an investigation-priority score, not a probability of failure.

In [ ]:
display(Image(filename=str(figures / 'phase9_critical_nodes.png')))

## Candidate Failure Propagation Routes

Beam search follows high-volume directed transitions from critical stations. The route score combines transition volume, labeled transition failure rate, and critical-node evidence. These routes should be validated with maintenance logs, sensor events, product family, and engineering knowledge before being described as physical failure propagation.

In [ ]:
display(routes.head(20))
display(Image(filename=str(figures / 'phase9_candidate_propagation_routes.png')))

## Graph Export And Future Use

`phase9_manufacturing_knowledge_graph.graphml` can be opened in Gephi, Cytoscape, or loaded with NetworkX. The node and edge CSV files are easier to query from the future Manufacturing Copilot and Executive Dashboard. The HTML report summarizes findings for technical stakeholders.

In [ ]:
report_path = reports / 'phase9_knowledge_graph_report.html'
graphml_path = reports / 'phase9_manufacturing_knowledge_graph.graphml'
print(report_path)
print(graphml_path)